Golden evaluation set: 15 verified questions with SQL and expected results for the UNDERWRITING_ANALYTICS semantic view
*Co-authored with CoCo*

# Golden Evaluation Set — Insurance Underwriting Analytics

This notebook contains **15 ground-truth question/SQL/result triples** for validating the `UNDERWRITING_ANALYTICS` semantic view.

| # | Complexity | Tables Involved |
|---|-----------|----------------|
| 1-3 | Simple | Single table, direct COUNT/filter |
| 4-6 | Simple-Medium | Single table, GROUP BY with aggregation |
| 7-10 | Medium | Multi-table JOINs, CASE expressions |
| 11-15 | Complex | Multi-table analytics, ratios, window functions |

---

## Q1 (Simple): How many policyholders are there in total?

**Expected:** 1,500

In [ ]:
%%sql -r q1_result
-- Q1: Total policyholder count
SELECT COUNT(*) AS TOTAL_POLICYHOLDERS
FROM INSURANCE_UNDERWRITING.RAW.RAW_POLICYHOLDERS;

## Q2 (Simple): How many policyholders are smokers?

**Expected:** 258

In [ ]:
%%sql -r q2_result
-- Q2: Smoker count
SELECT COUNT(*) AS SMOKER_COUNT
FROM INSURANCE_UNDERWRITING.RAW.RAW_POLICYHOLDERS
WHERE SMOKER_FLAG = TRUE;

## Q3 (Simple): What is the breakdown of policies by status?

**Expected:**
| POLICY_STATUS | POLICY_COUNT |
|---|---|
| Active | 2,252 |
| Lapsed | 392 |
| Expired | 201 |
| Pending | 96 |
| Cancelled | 59 |

In [ ]:
%%sql -r q3_result
-- Q3: Policies by status
SELECT POLICY_STATUS, COUNT(*) AS POLICY_COUNT
FROM INSURANCE_UNDERWRITING.RAW.RAW_POLICIES
GROUP BY POLICY_STATUS
ORDER BY POLICY_COUNT DESC;

## Q4 (Simple-Medium): What is the average annual premium by product line?

**Expected:**
| PRODUCT_LINE | AVG_ANNUAL_PREMIUM |
|---|---|
| Health | 7,955.02 |
| Term Life | 7,812.51 |
| Disability | 7,737.07 |
| Whole Life | 7,663.98 |
| Critical Illness | 7,397.90 |

In [ ]:
%%sql -r q4_result
-- Q4: Average annual premium by product line
SELECT PRODUCT_LINE, ROUND(AVG(ANNUAL_PREMIUM), 2) AS AVG_ANNUAL_PREMIUM
FROM INSURANCE_UNDERWRITING.RAW.RAW_POLICIES
GROUP BY PRODUCT_LINE
ORDER BY AVG_ANNUAL_PREMIUM DESC;

## Q5 (Simple-Medium): What is the distribution of underwriting decision outcomes?

**Expected:**
| DECISION_OUTCOME | DECISION_COUNT | PERCENTAGE |
|---|---|---|
| Approved - Standard | 2,041 | 68.03% |
| Approved - Rated | 804 | 26.80% |
| Approved - Exclusion | 142 | 4.73% |
| Deferred | 13 | 0.43% |

In [ ]:
%%sql -r q5_result
-- Q5: Underwriting decision outcome distribution
SELECT DECISION_OUTCOME,
       COUNT(*) AS DECISION_COUNT,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS PERCENTAGE
FROM INSURANCE_UNDERWRITING.RAW.RAW_UNDERWRITING_DECISIONS
GROUP BY DECISION_OUTCOME
ORDER BY DECISION_COUNT DESC;

## Q6 (Simple-Medium): What is premium collection performance by payment status?

**Expected:**
| PAYMENT_STATUS | CNT | TOTAL_DUE | TOTAL_PAID |
|---|---|---|---|
| Paid | 10,820 | 17,623,361.81 | 15,495,252.28 |
| Paid Late | 2,296 | 3,788,500.75 | 3,248,862.42 |
| Overdue | 362 | 622,566.06 | 539,721.08 |
| Missed | 14 | 16,273.36 | 15,171.82 |

In [ ]:
%%sql -r q6_result
-- Q6: Premium collection by payment status
SELECT PAYMENT_STATUS,
       COUNT(*) AS CNT,
       SUM(AMOUNT_DUE) AS TOTAL_DUE,
       SUM(AMOUNT_PAID) AS TOTAL_PAID
FROM INSURANCE_UNDERWRITING.RAW.RAW_PREMIUMS
GROUP BY PAYMENT_STATUS
ORDER BY CNT DESC;

## Q7 (Medium): What is the claim count and average claim amount by product line?

**Expected:**
| PRODUCT_LINE | CLAIM_COUNT | AVG_CLAIM_AMOUNT |
|---|---|---|
| Health | 210 | 75,772.50 |
| Term Life | 181 | 1,866,767.96 |
| Disability | 151 | 44,760.22 |
| Whole Life | 142 | 733,887.32 |
| Critical Illness | 116 | 274,650.21 |

In [ ]:
%%sql -r q7_result
-- Q7: Claims by product line (requires JOIN)
SELECT p.PRODUCT_LINE,
       COUNT(c.CLAIM_ID) AS CLAIM_COUNT,
       ROUND(AVG(c.CLAIM_AMOUNT), 2) AS AVG_CLAIM_AMOUNT
FROM INSURANCE_UNDERWRITING.RAW.RAW_CLAIMS c
JOIN INSURANCE_UNDERWRITING.RAW.RAW_POLICIES p ON c.POLICY_ID = p.POLICY_ID
GROUP BY p.PRODUCT_LINE
ORDER BY CLAIM_COUNT DESC;

## Q8 (Medium): What are the top 5 states by total sum assured?

**Expected:**
| STATE_CODE | POLICIES | TOTAL_SUM_ASSURED |
|---|---|---|
| FL | 352 | 268,452,000 |
| TX | 392 | 258,611,000 |
| CA | 356 | 238,457,000 |
| NY | 240 | 176,671,000 |
| NJ | 172 | 105,669,000 |

In [ ]:
%%sql -r q8_result
-- Q8: Top 5 states by total sum assured (requires JOIN)
SELECT ph.STATE_CODE,
       COUNT(DISTINCT pol.POLICY_ID) AS POLICIES,
       SUM(pol.SUM_ASSURED) AS TOTAL_SUM_ASSURED
FROM INSURANCE_UNDERWRITING.RAW.RAW_POLICIES pol
JOIN INSURANCE_UNDERWRITING.RAW.RAW_POLICYHOLDERS ph ON pol.POLICYHOLDER_ID = ph.POLICYHOLDER_ID
GROUP BY ph.STATE_CODE
ORDER BY TOTAL_SUM_ASSURED DESC
LIMIT 5;

## Q9 (Medium): What is the risk class distribution with average risk scores?

**Expected:**
| RISK_CLASS | COUNT | AVG_RISK_SCORE |
|---|---|---|
| Preferred Plus | 682 | 41.32 |
| Standard | 883 | 42.45 |
| Substandard | 169 | 42.76 |
| Super Preferred | 79 | 43.57 |
| Preferred | 1,177 | 44.26 |
| Decline | 10 | 52.12 |

In [ ]:
%%sql -r q9_result
-- Q9: Risk class distribution with avg scores
SELECT RISK_CLASS,
       COUNT(*) AS COUNT,
       ROUND(AVG(RISK_SCORE), 2) AS AVG_RISK_SCORE
FROM INSURANCE_UNDERWRITING.RAW.RAW_UNDERWRITING_DECISIONS
GROUP BY RISK_CLASS
ORDER BY AVG_RISK_SCORE;

## Q10 (Medium): How many claims are flagged as potential fraud and what is the total amount?

**Expected:** 22 fraud-flagged claims totalling $17,980,138.00

In [ ]:
%%sql -r q10_result
-- Q10: Fraud flagged claims
SELECT COUNT(*) AS FRAUD_CLAIM_COUNT,
       SUM(CLAIM_AMOUNT) AS TOTAL_FRAUD_AMOUNT
FROM INSURANCE_UNDERWRITING.RAW.RAW_CLAIMS
WHERE FRAUD_FLAG = TRUE;

## Q11 (Complex): What is the loss ratio by product line?

*Loss Ratio = Total Paid Claims / Total Premium Earned × 100*

**Expected:**
| PRODUCT_LINE | TOTAL_PREMIUM | TOTAL_PAID_CLAIMS | LOSS_RATIO_PCT |
|---|---|---|---|
| Term Life | 5,742,197.01 | 30,842,000.00 | 537.11% |
| Whole Life | 4,031,253.62 | 6,519,000.00 | 161.71% |
| Critical Illness | 3,528,796.55 | 488,267.46 | 13.84% |
| Disability | 3,922,695.53 | 457,465.01 | 11.66% |
| Health | 6,006,042.97 | 699,273.41 | 11.64% |

*Note: Life products have high loss ratios because sum assured is the full death benefit vs. one year's premium.*

In [ ]:
%%sql -r q11_result
-- Q11: Loss ratio by product line (multi-table with ratio calculation)
SELECT p.PRODUCT_LINE,
       SUM(p.ANNUAL_PREMIUM) AS TOTAL_PREMIUM,
       SUM(c.PAID_AMOUNT) AS TOTAL_PAID_CLAIMS,
       ROUND(SUM(c.PAID_AMOUNT) / NULLIF(SUM(p.ANNUAL_PREMIUM), 0) * 100, 2) AS LOSS_RATIO_PCT
FROM INSURANCE_UNDERWRITING.RAW.RAW_POLICIES p
LEFT JOIN INSURANCE_UNDERWRITING.RAW.RAW_CLAIMS c
    ON p.POLICY_ID = c.POLICY_ID AND c.CLAIM_STATUS = 'Paid'
GROUP BY p.PRODUCT_LINE
ORDER BY LOSS_RATIO_PCT DESC;

## Q12 (Complex): What is the average number of days from application to underwriting decision by risk class?

**Expected:**
| RISK_CLASS | AVG_DAYS_TO_DECISION | DECISIONS |
|---|---|---|
| Substandard | 12.3 | 169 |
| Super Preferred | 12.1 | 79 |
| Standard | 11.7 | 883 |
| Preferred Plus | 11.5 | 682 |
| Preferred | 11.4 | 1,177 |
| Decline | 11.0 | 10 |

In [ ]:
%%sql -r q12_result
-- Q12: Average days to decision by risk class (multi-table date arithmetic)
SELECT ud.RISK_CLASS,
       ROUND(AVG(DATEDIFF('day', p.APPLICATION_DATE, ud.DECISION_DATE)), 1) AS AVG_DAYS_TO_DECISION,
       COUNT(*) AS DECISIONS
FROM INSURANCE_UNDERWRITING.RAW.RAW_UNDERWRITING_DECISIONS ud
JOIN INSURANCE_UNDERWRITING.RAW.RAW_POLICIES p ON ud.POLICY_ID = p.POLICY_ID
GROUP BY ud.RISK_CLASS
ORDER BY AVG_DAYS_TO_DECISION DESC;

## Q13 (Complex): What is the premium collection rate by payment method?

**Expected:**
| PAYMENT_METHOD | TOTAL_PAYMENTS | SUCCESSFUL_PAYMENTS | COLLECTION_RATE_PCT |
|---|---|---|---|
| Wire Transfer | 2,751 | 2,682 | 97.49% |
| Check | 2,735 | 2,666 | 97.48% |
| Credit Card | 2,660 | 2,585 | 97.18% |
| Online Payment | 2,654 | 2,575 | 97.02% |
| ACH/Direct Debit | 2,692 | 2,608 | 96.88% |

In [ ]:
%%sql -r q13_result
-- Q13: Collection rate by payment method (CASE + ratio)
SELECT PAYMENT_METHOD,
       COUNT(*) AS TOTAL_PAYMENTS,
       SUM(CASE WHEN PAYMENT_STATUS IN ('Paid', 'Paid Late') THEN 1 ELSE 0 END) AS SUCCESSFUL_PAYMENTS,
       ROUND(SUM(CASE WHEN PAYMENT_STATUS IN ('Paid', 'Paid Late') THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS COLLECTION_RATE_PCT
FROM INSURANCE_UNDERWRITING.RAW.RAW_PREMIUMS
GROUP BY PAYMENT_METHOD
ORDER BY COLLECTION_RATE_PCT DESC;

## Q14 (Complex): How do smokers compare to non-smokers in terms of claims frequency?

**Expected:**
| SMOKER_FLAG | POLICYHOLDERS | AVG_INCOME | AVG_BMI | TOTAL_CLAIMS | CLAIMS_PER_HOLDER |
|---|---|---|---|---|---|
| FALSE | 1,242 | 139,343.33 | 29.9 | 651 | 0.524 |
| TRUE | 258 | 134,577.27 | 29.9 | 149 | 0.578 |

*Smokers have ~10% higher claims frequency (0.578 vs 0.524 claims per policyholder).*

In [ ]:
%%sql -r q14_result
-- Q14: Smoker vs non-smoker risk profile (3-table join with ratios)
SELECT ph.SMOKER_FLAG,
       COUNT(DISTINCT ph.POLICYHOLDER_ID) AS POLICYHOLDERS,
       ROUND(AVG(ph.ANNUAL_INCOME), 2) AS AVG_INCOME,
       ROUND(AVG(ph.BMI), 1) AS AVG_BMI,
       COUNT(DISTINCT c.CLAIM_ID) AS TOTAL_CLAIMS,
       ROUND(COUNT(DISTINCT c.CLAIM_ID) * 1.0 / COUNT(DISTINCT ph.POLICYHOLDER_ID), 3) AS CLAIMS_PER_HOLDER
FROM INSURANCE_UNDERWRITING.RAW.RAW_POLICYHOLDERS ph
LEFT JOIN INSURANCE_UNDERWRITING.RAW.RAW_CLAIMS c ON ph.POLICYHOLDER_ID = c.POLICYHOLDER_ID
GROUP BY ph.SMOKER_FLAG;

## Q15 (Complex): What is the end-to-end funnel from policies to claims by product line?

**Expected:**
| PRODUCT_LINE | TOTAL_POLICIES | APPROVED_POLICIES | CLAIMS_FILED | CLAIMS_PAID | CLAIM_FREQUENCY_PCT |
|---|---|---|---|---|---|
| Disability | 507 | 506 | 151 | 31 | 29.84% |
| Health | 755 | 750 | 210 | 49 | 28.00% |
| Whole Life | 526 | 522 | 142 | 36 | 27.20% |
| Term Life | 735 | 733 | 181 | 36 | 24.69% |
| Critical Illness | 477 | 476 | 116 | 30 | 24.37% |

*Claim frequency = claims filed / approved policies. Disability has the highest claim frequency at ~30%.*

In [ ]:
%%sql -r q15_result
-- Q15: End-to-end funnel (3-table join with conditional aggregation)
SELECT p.PRODUCT_LINE,
       COUNT(DISTINCT p.POLICY_ID) AS TOTAL_POLICIES,
       COUNT(DISTINCT CASE WHEN ud.DECISION_OUTCOME LIKE 'Approved%' THEN p.POLICY_ID END) AS APPROVED_POLICIES,
       COUNT(DISTINCT c.CLAIM_ID) AS CLAIMS_FILED,
       COUNT(DISTINCT CASE WHEN c.CLAIM_STATUS = 'Paid' THEN c.CLAIM_ID END) AS CLAIMS_PAID,
       ROUND(COUNT(DISTINCT c.CLAIM_ID) * 100.0 /
             NULLIF(COUNT(DISTINCT CASE WHEN ud.DECISION_OUTCOME LIKE 'Approved%' THEN p.POLICY_ID END), 0), 2) AS CLAIM_FREQUENCY_PCT
FROM INSURANCE_UNDERWRITING.RAW.RAW_POLICIES p
LEFT JOIN INSURANCE_UNDERWRITING.RAW.RAW_UNDERWRITING_DECISIONS ud ON p.POLICY_ID = ud.POLICY_ID
LEFT JOIN INSURANCE_UNDERWRITING.RAW.RAW_CLAIMS c ON p.POLICY_ID = c.POLICY_ID
GROUP BY p.PRODUCT_LINE
ORDER BY CLAIM_FREQUENCY_PCT DESC;

---
## Summary

| # | Question | Complexity | Tables |
|---|----------|-----------|--------|
| 1 | Total policyholders | Simple | Policyholders |
| 2 | Smoker count | Simple | Policyholders |
| 3 | Policies by status | Simple | Policies |
| 4 | Avg premium by product line | Simple-Medium | Policies |
| 5 | Decision outcome distribution | Simple-Medium | Underwriting |
| 6 | Premium collection by status | Simple-Medium | Premiums |
| 7 | Claims by product line | Medium | Claims + Policies |
| 8 | Top states by sum assured | Medium | Policies + Policyholders |
| 9 | Risk class distribution | Medium | Underwriting |
| 10 | Fraud flagged claims | Medium | Claims |
| 11 | Loss ratio by product line | Complex | Policies + Claims |
| 12 | Days to decision by risk class | Complex | Underwriting + Policies |
| 13 | Collection rate by payment method | Complex | Premiums |
| 14 | Smoker vs non-smoker claims | Complex | Policyholders + Claims |
| 15 | End-to-end policy funnel | Complex | Policies + Underwriting + Claims |